In [1]:
# Function 8 - BBO Capstone Project
# Strategy: Scaled Gaussian Process + EI/UCB Hybrid
# Function 8 is an 8D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 8 data
# --------------------------------------------------
# Use the same X and y arrays from your Function 8 data.

X = np.array([
    [0.60499445, 0.29221502, 0.90845275, 0.35550624, 0.20166872, 0.57533801, 0.31031095, 0.73428138],
    [0.17800696, 0.56622265, 0.99486184, 0.21032501, 0.32015266, 0.70790879, 0.63538449, 0.10713163],
    [0.00907698, 0.81162615, 0.52052036, 0.07568668, 0.26511183, 0.09165169, 0.59241515, 0.36732026],
    [0.50602816, 0.65373012, 0.36341078, 0.17798105, 0.09372830, 0.19742533, 0.75582690, 0.29247234],
    [0.35990926, 0.24907568, 0.49599717, 0.70921498, 0.11498719, 0.28920692, 0.55729515, 0.59388173],
    [0.77881834, 0.00341950, 0.33798313, 0.51952778, 0.82090699, 0.53724669, 0.55134710, 0.66003209],
    [0.90864932, 0.06224970, 0.23825955, 0.76660355, 0.13233596, 0.99024381, 0.68806782, 0.74249594],
    [0.58637144, 0.88073573, 0.74502075, 0.54603485, 0.00964888, 0.74899176, 0.23090707, 0.09791562],
    [0.76113733, 0.85467239, 0.38212433, 0.33735198, 0.68970832, 0.30985305, 0.63137968, 0.04195607],
    [0.98493320, 0.69950626, 0.99888550, 0.18014846, 0.58014315, 0.23108719, 0.49082694, 0.31368272],
    [0.11207131, 0.43773566, 0.59659878, 0.59277563, 0.22698177, 0.41010452, 0.92123758, 0.67475276],
    [0.79188751, 0.57619134, 0.69452836, 0.28342378, 0.13675546, 0.27916186, 0.84276726, 0.62532792],
    [0.14355030, 0.93741452, 0.23232482, 0.00904349, 0.41457893, 0.40932517, 0.55377852, 0.20584080],
    [0.76991655, 0.45875909, 0.55900044, 0.69460444, 0.50319902, 0.72834638, 0.78425353, 0.66313109],
    [0.05644741, 0.06595555, 0.02292868, 0.03878647, 0.40393544, 0.80105533, 0.48830701, 0.89308498],
    [0.86243745, 0.48273382, 0.28186940, 0.54410223, 0.88749026, 0.38265469, 0.60190199, 0.47646169],
    [0.35151190, 0.59006494, 0.90943630, 0.67840835, 0.21282566, 0.08846038, 0.41015300, 0.19572429],
    [0.73590364, 0.03461189, 0.72803027, 0.14742652, 0.29574314, 0.44511731, 0.97517969, 0.37433978],
    [0.68029397, 0.25510465, 0.86218799, 0.13439582, 0.32632920, 0.28790687, 0.43501048, 0.36420013],
    [0.04432925, 0.01358149, 0.25819824, 0.57764416, 0.05127992, 0.15856307, 0.59103012, 0.07795293],
    [0.77834548, 0.75114565, 0.31414221, 0.90298577, 0.33538166, 0.38632267, 0.74897249, 0.98875510],
    [0.89888711, 0.52364170, 0.87678325, 0.21869645, 0.90026089, 0.28276624, 0.91107791, 0.47239822],
    [0.14512029, 0.11932754, 0.42088822, 0.38760861, 0.15542283, 0.87517163, 0.51055967, 0.72861058],
    [0.33895442, 0.56693202, 0.37675110, 0.09891573, 0.65945169, 0.24554809, 0.76248278, 0.73215347],
    [0.17615002, 0.29396143, 0.97567997, 0.79393631, 0.92340076, 0.03084229, 0.80325452, 0.59589758],
    [0.02894663, 0.02827906, 0.48137155, 0.61317460, 0.67266045, 0.02211341, 0.60148330, 0.52488505],
    [0.19263987, 0.63067728, 0.41679584, 0.49052929, 0.79608602, 0.65456706, 0.27624119, 0.29551759],
    [0.94318502, 0.21885062, 0.72118408, 0.42459707, 0.98690200, 0.53518298, 0.71474318, 0.96009372],
    [0.53272140, 0.83369260, 0.07139900, 0.11681148, 0.73069311, 0.93737559, 0.86650798, 0.12790200],
    [0.44709584, 0.84395253, 0.72954612, 0.63915138, 0.40928714, 0.13264569, 0.03590888, 0.44683847],
    [0.38222497, 0.55713584, 0.85310163, 0.33379569, 0.26572127, 0.48087292, 0.23764706, 0.76863196],
    [0.53281953, 0.86230848, 0.53826712, 0.04944293, 0.71970119, 0.90670590, 0.10823094, 0.52534791],
    [0.39486519, 0.33180167, 0.74075430, 0.69786172, 0.73740444, 0.78377681, 0.25449546, 0.87114551],
    [0.98594539, 0.87305363, 0.07039262, 0.05358729, 0.73415296, 0.52025852, 0.81104004, 0.10336036],
    [0.96457339, 0.97397979, 0.66375335, 0.66221599, 0.67312167, 0.90523762, 0.45887462, 0.56091750],
    [0.47207071, 0.16820264, 0.08642757, 0.45265551, 0.48061922, 0.62243949, 0.92897446, 0.11253627],
    [0.85600695, 0.63889370, 0.32619202, 0.66850311, 0.24029837, 0.21029889, 0.16754636, 0.96358986],
    [0.81003174, 0.63504604, 0.26954758, 0.86960534, 0.66192159, 0.25225873, 0.76567003, 0.89054867],
    [0.79625252, 0.00703653, 0.35569738, 0.48756605, 0.74051962, 0.70665010, 0.99291449, 0.38173437],
    [0.48124533, 0.10246072, 0.21948594, 0.67732237, 0.24750919, 0.24434086, 0.16382453, 0.71596164],

    # Iteration results from our project
    [0.056447, 0.065955, 0.022928, 0.038786, 0.403935, 0.801055, 0.488307, 0.893084],
    [0.016826, 0.407406, 0.041439, 0.020643, 0.962136, 0.074516, 0.064114, 0.341289],
    [0.16880959, 0.16218930, 0.23082296, 0.16289581, 0.97818181, -0.26863235, 0.08418761, 0.34427606],
    [0.039675, 0.121395, 0.150160, 0.029602, 0.752700, 0.490718, 0.085935, 0.710564],
    [0.016906, 0.205707, 0.185215, 0.248031, 0.809195, 0.427772, 0.572305, 0.285428],
    [0.435346, 0.337494, 0.180133, 0.501014, 0.982053, 0.146277, 0.266897, 0.909149],
    [0.027543, 0.017589, 0.444975, 0.023136, 0.897680, 0.628403, 0.090747, 0.260113],
    [0.461943, 0.060764, 0.176900, 0.048671, 0.960098, 0.117481, 0.052454, 0.461135],
    [0.027277, 0.152897, 0.132831, 0.474030, 0.951068, 0.508039, 0.183118, 0.542624],
    [0.047108, 0.016209, 0.048751, 0.849189, 0.949619, 0.009065, 0.147757, 0.555296],
    [0.048262, 0.035632, 0.277898, 0.127328, 0.937781, 0.687998, 0.190591, 0.530371],
    [0.107653, 0.089220, 0.133346, 0.221671, 0.945634, 0.014570, 0.114405, 0.737282],
])

y = np.array([
    7.39872110, 7.00522736, 8.45948162, 8.28400781, 8.60611679,
    8.54174792, 7.32743458, 7.29987205, 7.95787474, 5.59219339,
    7.85454099, 6.79198578, 8.97655402, 7.37908290, 9.59848200,
    8.15998319, 7.13162397, 6.76796253, 7.43374407, 9.01307515,
    7.31089382, 5.84106731, 9.14163949, 8.81755844, 6.45194313,
    8.83074505, 9.34427428, 6.88784639, 8.04221254, 7.69236805,
    7.92375877, 8.42175924, 8.27806240, 7.11345716, 6.40258841,
    8.47293632, 7.97768459, 7.46087219, 7.43659353, 9.18300525,

    # Iteration outputs from our project
    9.5984813862679,
    9.6418401133519,
    9.8569503990799,
    9.9477396987374,
    9.6719546004781,
    9.4489820034974,
    9.6015622921061,
    9.5085606063475,
    9.8720202531314,
    9.2099667796019,
    9.8699339161814,
    9.7282337767126,
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format a point for BBO portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep candidate values inside [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates_scaled, model, scaler_y, y_best, xi=0.01):
    """Expected Improvement calculated on the original output scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates_scaled, model, scaler_y, kappa=2.0):
    """Upper Confidence Bound calculated on the original output scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]

    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)

if np.any(X < 0) or np.any(X >= 1):
    print("\nWarning: Some observed X values are outside [0, 1).")
    print("The model will use them as observed data, but new candidates are restricted to [0, 1).")


# --------------------------------------------------
# 4. Scale data
# --------------------------------------------------

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

print("\nAfter scaling:")
print(f"X_scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
print(f"y_scaled range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")


# --------------------------------------------------
# 5. Gaussian Process model
# --------------------------------------------------
# Function 8 is high-dimensional, so scaling and noise regularization are important.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3))
    * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(0.01, 1000.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e-1)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.01,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gpr.fit(X_scaled, y_scaled)

training_score = gpr.score(X_scaled, y_scaled)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training R²:", round(training_score, 4))

if training_score > 0.99:
    print("Warning: Training R² is very high. Check for possible overfitting.")


# --------------------------------------------------
# 6. Generate candidate points
# --------------------------------------------------
# Strategy:
# 30% local exploitation around the best point
# 70% global exploration because Function 8 is 8D.
# New candidates are restricted to the official BBO domain [0, 1).

np.random.seed(42)

best_point_clipped = np.clip(best_point, 0.0, 0.999999)

local_candidates = best_point_clipped + np.random.normal(
    loc=0.0,
    scale=0.12,
    size=(9000, X.shape[1])
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=X.shape[1], seed=42)
global_candidates = sampler.random(n=21000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])

X_candidates_scaled = scaler_X.transform(X_candidates)


# --------------------------------------------------
# 7. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    kappa=UCB_KAPPA
)

ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid score:
# 50% EI for improvement
# 50% UCB for uncertainty/exploration
hybrid_score = 0.50 * ei_norm + 0.50 * ucb_norm


# --------------------------------------------------
# 8. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

pred_scaled, std_scaled = gpr.predict(
    scaler_X.transform(x_next.reshape(1, -1)),
    return_std=True
)

predicted_y = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]
predicted_std = std_scaled[0] * scaler_y.scale_[0]

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y)
print("Uncertainty:", predicted_std)
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 9. Show top 5 candidates
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]

    pred_i_scaled, std_i_scaled = gpr.predict(
        scaler_X.transform(point.reshape(1, -1)),
        return_std=True
    )

    pred_i = scaler_y.inverse_transform(pred_i_scaled.reshape(-1, 1)).ravel()[0]
    std_i = std_i_scaled[0] * scaler_y.scale_[0]

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={pred_i:.6f}, "
        f"std={std_i:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )


# --------------------------------------------------
# 10. Sanity checks
# --------------------------------------------------

print("\nSanity Checks:")
print(f"Training y range: [{y.min():.3f}, {y.max():.3f}]")
print(f"Best observed y: {best_y:.6f}")
print(f"Number of candidate points: {len(X_candidates)}")
print(f"Candidate dimension: {X_candidates.shape[1]}")

Shape of X: (52, 8)
Shape of y: (52,)

Best point so far:
0.039675-0.121395-0.150160-0.029602-0.752700-0.490718-0.085935-0.710564
Best output so far: 9.9477396987374

The model will use them as observed data, but new candidates are restricted to [0, 1).

After scaling:
X_scaled range: [-2.372, 1.942]
y_scaled range: [-2.319, 1.493]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP Model Diagnostics:
Kernel: 4.67**2 * Matern(length_scale=[11.5, 15.8, 9.49, 19.1, 88.1, 1e+03, 13.9, 1e+03], nu=2.5) + WhiteKernel(noise_level=1e-06)
Training R²: 0.9972

Next Point to Sample:
X = [0.         0.         0.16418524 0.00495048 0.98926208 0.69070872
 0.18978    0.64016071]
Submission format: 0.000000-0.000000-0.164185-0.004950-0.989262-0.690709-0.189780-0.640161
Predicted y: 9.888351922994126
Uncertainty: 0.07631704858649528
Hybrid score: 0.9998983075485087

Top 5 Candidates:
1. X=0.000000-0.000000-0.164185-0.004950-0.989262-0.690709-0.189780-0.640161 | pred=9.888352, std=0.076317, score=0.999898
2. X=0.015147-0.000000-0.147428-0.000000-0.999999-0.542965-0.275501-0.846318 | pred=9.881914, std=0.080043, score=0.988666
3. X=0.000000-0.058281-0.181175-0.000000-0.999999-0.481739-0.160076-0.477561 | pred=9.897236, std=0.068282, score=0.965875
4. X=0.000000-0.000000-0.101554-0.000000-0.999999-0.602737-0.172311-0.604349 | pred=9.872456, std=0.084097, score=0.952368
5. X=0.00